# RQ1: Apache Tika (Standalone, Self-Contained)

Real extraction, matching, and code mining for Apache Tika, the third independent replication project. Includes the real, disclosed JMeter failure story and the resulting fail-fast JIRA verification safeguard. Output: `tika_real_mined_dataset.csv`.

In [1]:
!pip install -q pandas numpy requests lizard || pip install -q pandas numpy requests lizard --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00


In [2]:
"""
RQ1 SCOPE TEST : Apache Tika
================================================================================

"""
import subprocess
import os
import re
import time
import requests
import pandas as pd
import numpy as np
from collections import defaultdict

HEADERS = {"User-Agent": "qm640-capstone"}
np.random.seed(42)


def verify_project_uses_jira(project_key: str) -> bool:
    """Unchanged real safeguard: confirms the project has resolvable JIRA
    issues before running the full pipeline."""
    test_url = "https://issues.apache.org/jira/rest/api/2/search"
    params = {"jql": f"project={project_key}", "maxResults": 1}
    resp = requests.get(test_url, headers=HEADERS, timeout=15)
    if resp.status_code != 200:
        print(f"WARNING: real JIRA query for project={project_key} failed "
              f"(status {resp.status_code}). Stopping before running the full pipeline.")
        return False
    data = resp.json()
    total = data.get("total", 0)
    if total == 0:
        print(f"WARNING: real JIRA project={project_key} returned 0 total issues. Stopping.")
        return False
    print(f"Verified: real JIRA project={project_key} has {total} total real issues. Proceeding.")
    return True


def extract_tika_jira_issues():
    JIRA_BASE_URL = "https://issues.apache.org/jira/rest/api/2/search"
    PAGE_SIZE = 100
    all_issues = []
    start_at = 0
    jql = 'project=TIKA AND resolution=Fixed ORDER BY resolutiondate ASC'
    fields = "created,resolutiondate,priority,components,comment,summary,status,issuetype"

    while True:
        params = {"jql": jql, "startAt": start_at, "maxResults": PAGE_SIZE, "fields": fields}
        resp = requests.get(JIRA_BASE_URL, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        issues = data.get("issues", [])
        if not issues:
            break
        for issue in issues:
            f = issue["fields"]
            all_issues.append({
                "issue_id": issue["key"], "project_name": "TIKA",
                "created": f.get("created"), "resolution_date": f.get("resolutiondate"),
                "priority": (f.get("priority") or {}).get("name"),
                "num_comments": (f.get("comment") or {}).get("total", 0),
                "issue_type": (f.get("issuetype") or {}).get("name"),
            })
        start_at += PAGE_SIZE
        print(f"  TIKA: fetched {len(all_issues)} real issues so far...")
        time.sleep(0.5)
        if start_at >= data.get("total", 0):
            break

    df = pd.DataFrame(all_issues)
    df.to_csv("tika_jira_raw.csv", index=False)
    print(f"Real Tika JIRA extraction complete: {len(df)} real issues")
    return df


def ensure_tika_repo():
    path = "repos/tika"
    if not os.path.exists(path):
        print("Cloning real Apache Tika repository (full history, blobless)...")
        subprocess.run(["git", "clone", "--filter=blob:none", "https://github.com/apache/tika.git", path], check=True)
    else:
        print(f"Tika repository already present at {path}")
    return path


# ---------------------------------------------------------------------------

def build_issue_to_commit_index(repo_dir: str) -> dict:
    print("Reading full real Tika commit history for issue matching (one pass)...")
    result = subprocess.run(
        ["git", "-C", repo_dir, "log", "--all", "--format=%H|%s"],
        capture_output=True, text=True, timeout=300
    )
    index = defaultdict(list)
    pattern = re.compile(r"TIKA-\d+", re.IGNORECASE)
    for line in result.stdout.splitlines():
        if "|" not in line:
            continue
        sha, message = line.split("|", 1)
        for issue_id in pattern.findall(message):
            index[issue_id.upper()].append(sha)
    print(f"  Real index built: {len(index)} unique real Tika issue IDs found")
    return index


# ---------------------------------------------------------------------------

def build_commit_to_files_index(repo_dir: str) -> dict:
    print("Reading full real file-change history for Tika (one pass)...")
    result = subprocess.run(
        ["git", "-C", repo_dir, "log", "--all", "--name-only", "--format=COMMIT:%H"],
        capture_output=True, text=True, timeout=300
    )
    files_by_commit = defaultdict(list)
    current_sha = None
    for line in result.stdout.splitlines():
        if line.startswith("COMMIT:"):
            current_sha = line.replace("COMMIT:", "").strip()
        elif line.strip() and current_sha:
            files_by_commit[current_sha].append(line.strip())

    root_result = subprocess.run(
        ["git", "-C", repo_dir, "rev-list", "--max-parents=0", "--all"],
        capture_output=True, text=True, timeout=60
    )
    for sha in root_result.stdout.split():
        files_by_commit[sha] = []  # match diff-tree's real root-commit behavior

    print(f"  Real index built: {len(files_by_commit)} commits indexed")
    return files_by_commit


def get_changed_java_files_from_index(index: dict, sha: str) -> list:
    files = index.get(sha, [])
    files = [f for f in files if f.endswith(".java")]
    return [f for f in files if "/test/" not in f and "/generated/" not in f]


# ---------------------------------------------------------------------------

class BatchFileReader:
    """FIXED: same real bug found and fixed for Camel/Hadoop applied here --
    text-mode pipe + early return without consuming the response body could
    desync the stream and hang/crash on a non-blob (tree/commit) object."""

    def __init__(self, repo_dir: str):
        self.proc = subprocess.Popen(
            ["git", "-C", repo_dir, "cat-file", "--batch"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE
        )

    def read(self, sha: str, path: str) -> str:
        self.proc.stdin.write(f"{sha}:{path}\n".encode("utf-8"))
        self.proc.stdin.flush()
        header = self.proc.stdout.readline().decode("utf-8", errors="replace")
        parts = header.split()
        if len(parts) < 2 or parts[1] == "missing":
            return ""
        try:
            size = int(parts[2])
        except (ValueError, IndexError):
            return ""
        content_bytes = self.proc.stdout.read(size)
        self.proc.stdout.read(1)
        if parts[1] != "blob":
            return ""
        return content_bytes.decode("utf-8", errors="replace")

    def close(self):
        self.proc.stdin.close()
        self.proc.wait()


def analyze_commit_metrics(reader: "BatchFileReader", sha: str, files: list):
    import lizard
    total_nloc, total_complexity_sum, total_functions = 0, 0, 0
    files_analyzed = 0
    for path in files:
        content = reader.read(sha, path)
        if not content.strip():
            continue
        try:
            analysis = lizard.analyze_file.analyze_source_code(path, content)
        except Exception:
            continue
        total_nloc += analysis.nloc
        total_functions += len(analysis.function_list)
        for fn in analysis.function_list:
            total_complexity_sum += fn.cyclomatic_complexity
        files_analyzed += 1
    if files_analyzed == 0 or total_functions == 0:
        return None
    return {"loc": total_nloc, "cyclomatic_complexity": total_complexity_sum / total_functions,
            "num_functions": total_functions, "num_files_changed": files_analyzed}


def run_tika_scope_test(max_sample_per_era: int = 300, save_every: int = 25):
    if not verify_project_uses_jira("TIKA"):
        raise SystemExit("Stopping: real JIRA verification failed for project=TIKA.")

    jira_df = extract_tika_jira_issues()
    jira_df["resolution_date_parsed"] = pd.to_datetime(jira_df["resolution_date"], errors="coerce", utc=True)
    jira_df["era"] = (jira_df["resolution_date_parsed"] >= "2023-01-01").map({True: "ai_era", False: "pre_ai"})

    repo_dir = ensure_tika_repo()


    commit_index = build_issue_to_commit_index(repo_dir)
    real_issue_ids = set(jira_df["issue_id"])
    matched_ids = real_issue_ids & set(commit_index.keys())
    print(f"Real match rate: {len(matched_ids)} of {len(real_issue_ids)} "
          f"({len(matched_ids)/max(len(real_issue_ids),1)*100:.1f}%) Tika issues have a real matching commit")

    matchable = jira_df[jira_df["issue_id"].isin(matched_ids)]

    samples = []
    for era in ["pre_ai", "ai_era"]:
        cell = matchable[matchable.era == era]
        n = min(max_sample_per_era, len(cell))
        if n > 0:
            samples.append(cell.sample(n=n, random_state=42))
        print(f"Real {era} matchable pool: {len(cell)}, sampling {n}")

    if not samples:
        print("No real matchable issues found in either era. Reporting this honestly.")
        return None

    sample_df = pd.concat(samples, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)


    files_index = build_commit_to_files_index(repo_dir)
    reader = BatchFileReader(repo_dir)

    rows = []
    for i, (_, row) in enumerate(sample_df.iterrows()):
        shas = commit_index.get(row["issue_id"].upper(), [])  # instant lookup
        if not shas:
            continue
        sha = shas[0]
        files = get_changed_java_files_from_index(files_index, sha)  # instant lookup
        if not files:
            continue
        metrics = analyze_commit_metrics(reader, sha, files)  # batched, no new subprocess
        if metrics is None:
            continue
        metrics.update({"issue_id": row["issue_id"], "project_name": "TIKA", "era": row["era"],
                         "issue_type": row["issue_type"], "defect_prone": 1 if row["issue_type"] == "Bug" else 0})
        rows.append(metrics)
        if (i + 1) % save_every == 0:
            pd.DataFrame(rows).to_csv("tika_mining_progress.csv", index=False)
            print(f"  [{i+1}/{len(sample_df)}] processed, {len(rows)} real samples so far")

    reader.close()

    out = pd.DataFrame(rows)
    out.to_csv("tika_real_mined_dataset.csv", index=False)
    print(f"\nTIKA SCOPE TEST COMPLETE: {len(out)} real mined code samples")
    print(f"\nSend me: tika_jira_raw.csv AND tika_real_mined_dataset.csv")
    return out


if __name__ == "__main__":
    result = run_tika_scope_test(max_sample_per_era=300)


Verified: real JIRA project=TIKA has 1172069 total real issues. Proceeding.
  TIKA: fetched 100 real issues so far...
  TIKA: fetched 200 real issues so far...
  TIKA: fetched 300 real issues so far...
  TIKA: fetched 400 real issues so far...
  TIKA: fetched 500 real issues so far...
  TIKA: fetched 600 real issues so far...
  TIKA: fetched 700 real issues so far...
  TIKA: fetched 800 real issues so far...
  TIKA: fetched 900 real issues so far...
  TIKA: fetched 1000 real issues so far...
  TIKA: fetched 1100 real issues so far...
  TIKA: fetched 1200 real issues so far...
  TIKA: fetched 1300 real issues so far...
  TIKA: fetched 1400 real issues so far...
  TIKA: fetched 1500 real issues so far...
  TIKA: fetched 1600 real issues so far...
  TIKA: fetched 1700 real issues so far...
  TIKA: fetched 1800 real issues so far...
  TIKA: fetched 1900 real issues so far...
  TIKA: fetched 2000 real issues so far...
  TIKA: fetched 2100 real issues so far...
  TIKA: fetched 2200 real issu